# Aligned feature extraction — ~241 features

Consumes `01-alignment.ipynb` outputs and writes per-patient JSON + a merged CSV.

Categories: whole density, zone density (ETDRS + Wong), skeleton, caliber, Knudtson, color, tortuosity, FAZ, AV crossings, bifurcation, width variability, junction exponent.

## 0. Setup — imports and paths

In [1]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from features.zones import (
    make_etdrs_9subfields, make_disc_zones, mm_per_px_from_disc,
    ETDRS_9_NAMES, DISC_ZONE_NAMES,
    DEFAULT_ETDRS_RADII_MM, DEFAULT_DISC_MARGIN_ZONES,
)
from features.density  import zone_density_features, whole_image_density_features
from features.skeleton import (
    whole_image_skeleton_features, whole_image_segment_features,
    whole_image_tortuosity_features, whole_image_bifurcation_features,
)
from features.caliber  import (
    whole_image_caliber_features, compute_crae_crve_avr,
    whole_image_width_variability, whole_image_junction_features,
)
from features.topology import whole_image_fractal_features, sholl_features
from features.color    import whole_image_color_features
from features.faz      import compute_faz
from features.crossings import crossings_features

In [2]:
SQUARE_SIZE  = 512
OUTPUT_ROOT  = PROJECT_ROOT / 'samples' / 'output'

rgb_path     = OUTPUT_ROOT / f'preprocessed_{SQUARE_SIZE}' / 'rgb'
aligned_root = OUTPUT_ROOT / f'aligned_{SQUARE_SIZE}'
features_out = OUTPUT_ROOT / f'features_json_aligned_{SQUARE_SIZE}'
features_out.mkdir(parents=True, exist_ok=True)

inv_df  = pd.read_csv(aligned_root / 'inventory.csv', parse_dates=['date'])
reg_log = pd.read_csv(aligned_root / 'registration_log.csv', parse_dates=['date'])

inventory = {}
for (pid, eye), grp in inv_df.groupby(['patient', 'eye'], sort=False):
    inventory.setdefault(pid, {})[eye] = grp.sort_values(['date', 'shot']).to_dict('records')

theta_cols = [f't{i}{j}' for i in range(3) for j in range(3)]
def get_theta(image_id):
    row = reg_log[reg_log['id'] == image_id]
    if len(row) == 0:
        return None
    return row.iloc[0][theta_cols].values.astype(np.float64).reshape(3, 3)

def warp_image(img, theta, H, W, is_mask=False):
    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    return cv2.warpAffine(img, theta[:2, :], (W, H),
                          flags=flags, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

patient_ids = sorted(inventory.keys())
for pid in patient_ids:
    for eye, visits in inventory[pid].items():
        print(f'{pid}/{eye}: {len(visits)} visits')

SAMPLE/R: 3 visits


## 1. Zones + common-valid mask per group

Zones built once from the fixed-visit geometry (same across visits). `common_valid` = fundus-pixel intersection across all visits of the group.

In [3]:
zones_per_group = {}
zone_meta = []
for pid, by_eye in inventory.items():
    for eye, visits in by_eye.items():
        if eye == 'unknown' or len(visits) < 1:
            continue
        fixed = visits[0]
        fx, fy = fixed['x_fovea'], fixed['y_fovea']
        dcx, dcy = fixed['x_disc'], fixed['y_disc']
        grp_df = inv_df[(inv_df['patient'] == pid) & (inv_df['eye'] == eye)]
        dd_px = grp_df['disc_diameter'].mean()
        mpp = mm_per_px_from_disc(dd_px)
        etdrs  = make_etdrs_9subfields(fx, fy, eye, mpp, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_ETDRS_RADII_MM)
        disc_z = make_disc_zones(dcx, dcy, dd_px, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_DISC_MARGIN_ZONES)
        zones_per_group[(pid, eye)] = {**etdrs, **disc_z}
        zone_meta.append({
            'patient': pid, 'eye': eye,
            'fovea_x': fx, 'fovea_y': fy,
            'disc_cx': dcx, 'disc_cy': dcy,
            'disc_diameter_px': dd_px, 'mm_per_px': mpp,
        })
zone_meta_df = pd.DataFrame(zone_meta)
zone_meta_df

,patient,eye,fovea_x,fovea_y,disc_cx,disc_cy,disc_diameter_px,mm_per_px
0,SAMPLE,R,360.5,265.299988,168.424698,237.091275,68.353206,0.026334


In [4]:
def fundus_valid(rgb):
    return rgb.sum(-1) > 10

def compute_common_valid(visits):
    common = np.ones((SQUARE_SIZE, SQUARE_SIZE), dtype=bool)
    for i, v in enumerate(visits):
        rgb = np.array(Image.open(rgb_path / f'{v["id"]}.png'))
        if i == 0:
            common &= fundus_valid(rgb)
        else:
            theta = get_theta(v['id'])
            if theta is None:
                continue
            common &= fundus_valid(warp_image(rgb, theta, SQUARE_SIZE, SQUARE_SIZE))
    return common

common_valid_per_group = {
    key: compute_common_valid(inventory[key[0]][key[1]])
    for key in zones_per_group.keys()
}
print(f'common_valid computed for {len(common_valid_per_group)} group(s)')

common_valid computed for 1 group(s)


## 2. Feature extraction — all 12 categories in a single loop

In [5]:
feat_rows = []
for pid, by_eye in tqdm(inventory.items(), desc='patients'):
    for eye, visits in by_eye.items():
        if eye == 'unknown':
            continue
        key = (pid, eye)
        if key not in zones_per_group:
            continue
        zones    = zones_per_group[key]
        valid    = common_valid_per_group[key]
        meta     = zone_meta_df.query('patient == @pid & eye == @eye').iloc[0]
        mpp_grp  = meta['mm_per_px']
        dd_grp   = meta['disc_diameter_px']
        fx, fy   = meta['fovea_x'], meta['fovea_y']
        dcx, dcy = meta['disc_cx'], meta['disc_cy']
        fixed_id = visits[0]['id']
        aligned  = aligned_root / str(pid) / eye

        for v in visits:
            vid = v['id']
            try:
                rgb_base = np.array(Image.open(rgb_path / f'{vid}.png'))
                vs_m = np.array(Image.open(aligned / 'vessels' / f'{vid}.png'))
                av_m = np.array(Image.open(aligned / 'av'      / f'{vid}.png'))
                dc_m = np.array(Image.open(aligned / 'discs'   / f'{vid}.png'))
            except FileNotFoundError:
                continue

            dd_v = v.get('disc_diameter', np.nan)
            if np.isfinite(dd_v) and dd_v >= 5:
                mpp_v = mm_per_px_from_disc(dd_v)
            else:
                dd_v, mpp_v = float(dd_grp), float(mpp_grp)

            if vid == fixed_id:
                rgb_a = rgb_base
            else:
                theta = get_theta(vid)
                rgb_a = warp_image(rgb_base, theta, SQUARE_SIZE, SQUARE_SIZE) if theta is not None else rgb_base

            # Existing 6 categories
            w_dens   = whole_image_density_features(vs_m, av_m, dc_m, valid_mask=valid)
            z_dens   = zone_density_features(vs_m, av_m, dc_m, zones, valid_mask=valid)
            w_skel   = whole_image_skeleton_features(vs_m, av_m, valid_mask=valid)
            w_seg    = whole_image_segment_features(vs_m, av_m, valid_mask=valid)
            w_cal    = whole_image_caliber_features(vs_m, av_m, valid_mask=valid, mm_per_px=mpp_v)
            knudtson = compute_crae_crve_avr(av_m, zones['zone_B'], mm_per_px=mpp_v)
            w_frac   = whole_image_fractal_features(vs_m, av_m, valid_mask=valid)
            w_sh_f   = sholl_features(vs_m, av_m, (fx, fy),   valid_mask=valid, center_name='fovea')
            w_sh_d   = sholl_features(vs_m, av_m, (dcx, dcy), valid_mask=valid, center_name='disc')
            w_color  = whole_image_color_features(rgb_a, vs_m, av_m, valid_mask=valid)

            # New Tier 1 + 2
            w_tort   = whole_image_tortuosity_features(vs_m, av_m, valid_mask=valid)
            faz      = compute_faz(vs_m, (fx, fy), valid_mask=valid, mm_per_px=mpp_v, disc_diameter_px=dd_v)
            cross    = crossings_features(av_m, zones=zones, valid_mask=valid)
            w_bif    = whole_image_bifurcation_features(vs_m, av_m, valid_mask=valid)
            w_wv     = whole_image_width_variability(vs_m, av_m, valid_mask=valid)
            w_junc   = whole_image_junction_features(vs_m, av_m, valid_mask=valid)

            feat_rows.append({
                'id': vid, 'patient': pid, 'eye': eye,
                'date': v['date'], 'shot': v['shot'],
                'disc_diameter_px': dd_v, 'mm_per_px': mpp_v,
                **w_dens, **z_dens, **w_skel, **w_seg, **w_cal, **knudtson,
                **w_frac, **w_sh_f, **w_sh_d, **w_color,
                **w_tort, **faz, **cross, **w_bif, **w_wv, **w_junc,
            })

features_zones = pd.DataFrame(feat_rows).sort_values(['patient', 'eye', 'date']).reset_index(drop=True)
print(f'extracted features for {len(features_zones)} visits × {features_zones.shape[1]} columns')

patients: 100%|██████████| 1/1 [00:02<00:00,  2.73s/it]

extracted features for 3 visits × 248 columns


## 3. Save — per-patient JSON + merged CSV

In [6]:
def _jd(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        v = float(o)
        return None if not np.isfinite(v) else v
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, pd.Timestamp):
        return o.isoformat()
    return str(o)

n_saved = 0
for pid, pdf in features_zones.groupby('patient', sort=False):
    eye_meta = {}
    for eye in pdf['eye'].unique():
        mrow = zone_meta_df.query('patient == @pid & eye == @eye')
        if len(mrow) == 0:
            continue
        m = mrow.iloc[0]
        eye_meta[eye] = {
            'fovea_x': float(m['fovea_x']), 'fovea_y': float(m['fovea_y']),
            'disc_cx': float(m['disc_cx']), 'disc_cy': float(m['disc_cy']),
            'disc_diameter_px': float(m['disc_diameter_px']),
            'mm_per_px': float(m['mm_per_px']),
            'n_visits': int((pdf['eye'] == eye).sum()),
        }
    payload = {
        'patient': str(pid),
        'config': {
            'mode': 'aligned', 'square_size': SQUARE_SIZE,
            'extracted_at': datetime.now(timezone.utc).isoformat(),
        },
        'meta_by_eye': eye_meta,
        'features': pdf.assign(date=pdf['date'].astype(str)).to_dict('records'),
    }
    with open(features_out / f'{pid}.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=_jd)
    n_saved += 1

merged_csv = features_out / '_merged_features_aligned.csv'
features_zones.to_csv(merged_csv, index=False)
print(f'saved: {n_saved} patient JSON file(s) → {features_out}')
print(f'       + {merged_csv.name} (rows={len(features_zones)}, cols={features_zones.shape[1]})')
features_zones.head()

saved: 1 patient JSON file(s) → D:\fundus-vessel-features\samples\output\features_json_aligned_512
       + _merged_features_aligned.csv (rows=3, cols=248)


,id,patient,eye,date,shot,disc_diameter_px,mm_per_px,whole_vessel_density,whole_artery_density,whole_vein_density,...,whole_vein_width_cov_max,whole_vessel_junc_X_mean,whole_vessel_junc_X_std,whole_vessel_junc_n,whole_artery_junc_X_mean,whole_artery_junc_X_std,whole_artery_junc_n,whole_vein_junc_X_mean,whole_vein_junc_X_std,whole_vein_junc_n
0,pid_1,SAMPLE,R,2020-01-15,1,68.868115,0.026137,0.104145,0.047361,0.057977,...,0.327540,1.871233,0.537875,73,1.872727,0.699469,11,2.011765,0.699481,17
1,pid_2,SAMPLE,R,2021-01-20,1,67.768540,0.026561,0.098376,0.043010,0.053914,...,0.325196,1.741176,0.423478,68,1.903846,0.691281,13,1.680000,0.334440,20
2,pid_3,SAMPLE,R,2021-07-05,1,68.422963,0.026307,0.103163,0.046795,0.056459,...,0.423749,1.892647,0.759947,68,1.903571,0.364723,14,1.858824,0.580880,17
